# Submit Training Job to Azure ML

**Previous:** [01-prepare-data.ipynb](01-prepare-data.ipynb) | **Next:** [03-download-trained-model.ipynb](03-download-trained-model.ipynb)

---

This notebook submits a **remote training job** to Azure ML compute cluster for distributed fine-tuning.

## What This Notebook Does

1. **Validates Prerequisites**: Ensures compute cluster and data are ready
2. **Packages Training Code**: Bundles your training scripts for remote execution
3. **Configures Job**: Sets up environment, compute, and training parameters
4. **Submits to Azure ML**: Starts the training job on the compute cluster - the model will be automatically downloaded from Azure AI Foundry during training
5. **Monitors Progress**: Tracks job status and training metrics in real-time
6. **Handles Checkpoints**: Saves model checkpoints to Azure ML datastore

## Prerequisites

- Infrastructure provisioned (via Terraform/azd) including compute cluster
- Training data uploaded to Azure ML datastore (notebook 01-prepare-data.ipynb)
- Sufficient GPU quota in Azure subscription

**Note**: The base model will be automatically downloaded from Azure AI Foundry during the remote training job. No local download required.

## Job Configuration

Training jobs are configured via:
- `configs/training_config.yaml`: Training hyperparameters and model ID
- `configs/model_config.yaml`: Model architecture settings
- `configs/data_config.yaml`: Data paths and preprocessing

## Expected Duration

- **Job Submission**: ~2-5 minutes
- **Cluster Warm-up**: ~5-10 minutes (if scaling from 0)
- **Model Download**: ~5-10 minutes (first time, cached afterwards)
- **Training**: 15-90 minutes (depending on dataset size and GPU)

## Cost Estimation

Approximate costs (varies by region):
- Standard_NC6s_v3 (1x V100): ~$3/hour
- Standard_NC12s_v3 (2x V100): ~$6/hour
- Standard_NC24s_v3 (4x V100): ~$12/hour

**Cost Optimization Tips:**
- Use auto-scaling (min_instances=0)
- Submit jobs in batches
- Use spot/low-priority instances
- Stop jobs if validation loss plateaus early

## 1. Setup Azure ML Workspace

**Why use JobManager?** The `AzureMLJobManager` class abstracts Azure ML SDK complexity, handling environment creation, job configuration, script packaging, and error handling with sensible defaults.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.training.job_manager import AzureMLJobManager
from src.utils.config import load_config
import yaml

# Load configuration (handles .env loading automatically)
config = load_config()

print(f"✓ Configuration loaded")
print(f"  Subscription: {config.azure.subscription_id}")
print(f"  Resource Group: {config.azure.resource_group}")
print(f"  Workspace: {config.azure.workspace_name}")

# Initialize job manager with configuration
job_manager = AzureMLJobManager(
    subscription_id=config.azure.subscription_id,
    resource_group=config.azure.resource_group,
    workspace_name=config.azure.workspace_name,
)

ml_client = job_manager.ml_client

print(f"✓ Connected to workspace: {ml_client.workspace_name}")
print("✓ Job manager initialized")

## 2. Verify Training Configuration

In [ ]:
# Load training configuration
training_config_path = project_root / "configs" / "training_config.yaml"

with open(training_config_path) as f:
    training_config = yaml.safe_load(f)

print("Training Configuration:")
print(f"  Model: {training_config['model']['name_or_path']}")
print(f"  Epochs: {training_config['training']['num_epochs']}")
print(f"  Batch size: {training_config['training']['per_device_train_batch_size']}")
print(f"  Learning rate: {training_config['training']['learning_rate']}")
print(f"  LoRA rank: {training_config['lora']['r']}")
print(f"  LoRA alpha: {training_config['lora']['alpha']}")
print(f"\nCompute:")
# Use compute name from config loaded in step 1 (from .env)
compute_name = config.azure.compute_name
print(f"  Cluster: {compute_name}")

## 3. Create Training Environment

**Why create environment?** Azure ML environments define the Docker container with all dependencies (PyTorch, CUDA, transformers) that runs on the compute cluster. We version environments to ensure reproducibility.

In [ ]:
# Create or update Azure ML environment
environment_name = "phi4-training-env"
conda_file = str(project_root / "configs" / "conda.yaml")

print(f"Creating environment: {environment_name}")
environment = job_manager.create_environment(
    name=environment_name,
    conda_file=conda_file,
    description="PyTorch environment for Phi-4 fine-tuning with LoRA",
)

print(f"✓ Environment created: {environment.name}")
print(f"  Version: {environment.version}")

## 4. Verify Compute Cluster

In [ ]:
# Check compute cluster status
try:
    compute = ml_client.compute.get(compute_name)
    print(f"✓ Compute cluster found: {compute.name}")
    print(f"  Type: {compute.type}")
    print(f"  Size: {compute.size}")
    print(f"  State: {compute.provisioning_state}")
    print(f"  Current nodes: {compute.current_node_count if hasattr(compute, 'current_node_count') else 'N/A'}")
except Exception as e:
    print(f"⚠️  Compute cluster not found: {e}")
    print(f"Expected cluster name: {compute_name}")
    print("\nThis compute cluster should have been provisioned by Terraform/azd.")
    print("Please check your infrastructure deployment or update AZUREML_COMPUTE_NAME in .env")

## 5. Verify Training Data & Submit Job

**Important:** Before submitting, ensure you've completed notebook `01-prepare-data.ipynb` to upload training data to Azure blob storage. The training job will automatically mount the data from the configured datastore.

In [ ]:
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential

print("🔍 Checking for training data in datastores...\n")

# Check both possible datastores
datastores_to_check = ["training_data_store", "workspaceblobstore"]
data_found_in = None

for ds_name in datastores_to_check:
    try:
        datastore = ml_client.datastores.get(ds_name)
        print(f"📦 Datastore: {ds_name}")
        print(f"   Account: {datastore.account_name}")
        print(f"   Container: {datastore.container_name}")

        # Try to access the blob storage
        credential = DefaultAzureCredential()
        blob_service_client = BlobServiceClient(
            account_url=f"https://{datastore.account_name}.blob.core.windows.net",
            credential=credential
        )
        container_client = blob_service_client.get_container_client(datastore.container_name)

        # Check for training data files at container root
        # (they should be train.jsonl and val.jsonl, not in a subfolder)
        files_to_check = ["train.jsonl", "val.jsonl"]
        files_found = []

        for file_path in files_to_check:
            blob_client = container_client.get_blob_client(file_path)
            try:
                props = blob_client.get_blob_properties()
                files_found.append(file_path)
                size_mb = props.size / (1024 * 1024)
                print(f"   ✓ {file_path} ({size_mb:.2f} MB)")
            except Exception:
                # File not found at root, try training-data/ subfolder (old location)
                try:
                    alt_path = f"training-data/{file_path}"
                    blob_client = container_client.get_blob_client(alt_path)
                    props = blob_client.get_blob_properties()
                    files_found.append(alt_path)
                    size_mb = props.size / (1024 * 1024)
                    print(f"   ✓ {alt_path} ({size_mb:.2f} MB) [legacy path]")
                except Exception:
                    pass

        if len(files_found) == len(files_to_check):
            data_found_in = ds_name
            # Check if using legacy paths
            using_legacy = any("/" in f for f in files_found)
            if using_legacy:
                print(f"   ⚠️  Data found in legacy location (training-data/ subfolder)")
                print(f"   💡 Run notebook 01 cell 8 again to fix the path")
            else:
                print(f"   ✅ All training data files found!\n")
            break
        elif files_found:
            print(f"   ⚠️  Some files missing\n")
        else:
            print(f"   ✗ No training data found\n")

    except Exception as e:
        print(f"   ✗ Could not access datastore: {e}\n")

if data_found_in:
    print(f"✅ Training data is ready in datastore: {data_found_in}")
    print(f"   You can proceed with job submission.")
else:
    print(f"⚠️  WARNING: Training data not found in any datastore!")
    print(f"   Please run notebook 01-prepare-data.ipynb to:")
    print(f"   1. Upload data to blob storage")
    print(f"   2. Register the datastore with Azure ML")

## 4a. Verify Training Data Availability

Before submitting the job, let's verify that the training data exists in blob storage.

In [ ]:
from datetime import datetime
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

# Configure job parameters
experiment_name = training_config.get('azure_ml', {}).get('experiment_name', 'phi-4-training')
display_name = f"phi4-finetuning-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Try to find the datastore with our training data
# First check for custom datastore created in notebook 01
datastore_name = None
datastore_options = ["training_data_store", "workspaceblobstore"]

print("🔍 Looking for datastore with training data...")
for ds_name in datastore_options:
    try:
        ds = ml_client.datastores.get(ds_name)
        print(f"  ✓ Found datastore: {ds_name}")
        print(f"    Account: {ds.account_name}")
        print(f"    Container: {ds.container_name}")
        datastore_name = ds_name
        break
    except Exception:
        print(f"  ✗ Datastore '{ds_name}' not found")

if not datastore_name:
    print(f"\n⚠️  WARNING: No suitable datastore found!")
    print(f"    Using fallback: workspaceblobstore")
    print(f"    If job fails, run notebook 01-prepare-data.ipynb to register datastore")
    datastore_name = "workspaceblobstore"

# Configure data inputs from blob storage
# Files should be at container root: train.jsonl and val.jsonl
data_inputs = {
    "training_data": Input(
        type=AssetTypes.URI_FOLDER,
        path=f"azureml://datastores/{datastore_name}/paths/",
    )
}

# Command to run - files will be mounted at ${{inputs.training_data}}/
command_str = """
python -m src.training.train \
  --config configs/training_config.yaml \
  --train-file ${{inputs.training_data}}/train.jsonl \
  --validation-file ${{inputs.training_data}}/val.jsonl
"""

print(f"\n📚 Submitting new training job...")
print(f"  Experiment: {experiment_name}")
print(f"  Display name: {display_name}")
print(f"  Compute: {compute_name}")
print(f"  Environment: {environment_name}")
print(f"\n📁 Data Inputs:")
print(f"  Datastore: {datastore_name}")
print(f"  Container root path: /")
print(f"  Files: train.jsonl, val.jsonl")
print(f"  Full URI: azureml://datastores/{datastore_name}/paths/")
print(f"\nCommand: {command_str.strip()}")
print("\n" + "="*60)

# Submit job with data inputs
job = job_manager.submit_training_job(
    experiment_name=experiment_name,
    display_name=display_name,
    code_path=str(project_root),
    command_str=command_str,
    environment_name=environment_name,
    compute_name=compute_name,
    environment_variables={
        "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:512",
    },
    inputs=data_inputs,
)

print(f"\n✓ Job submitted successfully!")
print(f"  Job ID: {job.name}")
print(f"  Status: {job.status}")
print(f"\n📊 View in Azure ML Studio:")
print(f"  {job.studio_url}")
print(f"\n💡 Note: Training data will be automatically mounted from blob storage")

# Store job name for later cells
job_name = job.name

## 6. Monitor Training Progress

In [ ]:
# Monitor all relevant jobs - comprehensive status view
import pandas as pd

# Try to get job_name from previous cell, or find most recent active job
if 'job_name' not in locals() or not job_name:
    print("⚠️  job_name not found. Looking for most recent active job...")
    experiment_name = training_config.get('azure_ml', {}).get('experiment_name', 'phi-4-training')
    recent_jobs = list(job_manager.list_jobs(experiment_name=experiment_name, max_results=5))

    if recent_jobs:
        job_name = recent_jobs[0].name
        print(f"✓ Using most recent job: {job_name}\n")
    else:
        print("❌ No jobs found. Please run the 'Submit Training Job' cell first.")
        job_name = None

print(f"📊 Training Progress Monitor")
print("="*80)

# === Environment Build Status ===
print(f"\n🔧 Environment Build Status (prepare_image experiment):")
print("-"*80)
env_build_active = False
try:
    # Filter jobs by experiment name after retrieval
    all_jobs = list(ml_client.jobs.list(list_view_type="All"))
    env_jobs = [j for j in all_jobs if hasattr(j, 'experiment_name') and j.experiment_name == "prepare_image"][:2]

    if env_jobs:
        for env_job in env_jobs:
            status_emoji = "🔄" if env_job.status in ['Running', 'Starting', 'Queued'] else "✅" if env_job.status == 'Completed' else "❌" if env_job.status == 'Failed' else "⏸️"
            print(f"{status_emoji} {env_job.display_name}")
            print(f"  Status: {env_job.status}")
            print(f"  Created: {env_job.creation_context.created_at.strftime('%Y-%m-%d %H:%M:%S')}")

            if env_job.status in ['Running', 'Starting', 'Queued']:
                env_build_active = True
                print(f"  ⚠️  Training jobs waiting for this")
            print(f"  Studio: {env_job.studio_url}")
            print()
    else:
        print("  No environment build jobs found")
except Exception as e:
    print(f"  Unable to query: {e}")

# === Current Training Job Status ===
if job_name:
    print(f"\n📚 Current Training Job Status:")
    print("-"*80)
    try:
        status = job_manager.get_job_status(job_name)
        status_emoji = "🔄" if status['status'] in ['Running', 'Preparing', 'Starting'] else "✅" if status['status'] == 'Completed' else "❌"

        print(f"{status_emoji} Job: {status['name']}")
        print(f"  Status: {status['status']}")
        print(f"  Created: {status['creation_time']}")
        print(f"  Duration: {status['duration']}")

        if status['status'] == 'Preparing' and env_build_active:
            print(f"  💡 Waiting: Environment image is still building (see above)")
        elif status['status'] == 'Preparing':
            print(f"  💡 Preparing: Allocating compute resources")
        elif status['status'] == 'Running':
            print(f"  ✅ Training in progress")

        print(f"  Studio: {status['studio_url']}")
    except Exception as e:
        print(f"  Unable to get status: {e}")

# === All Training Jobs Summary ===
print(f"\n📋 All Training Jobs ('{experiment_name}' experiment):")
print("-"*80)
jobs = job_manager.list_jobs(experiment_name=experiment_name, max_results=10)

if jobs:
    jobs_data = []
    for j in jobs:
        created = j.creation_context.created_at

        # Calculate duration
        duration_str = "N/A"
        if hasattr(j, 'creation_context') and j.creation_context.last_modified_at:
            try:
                duration = j.creation_context.last_modified_at - created
                duration_str = str(duration).split('.')[0]
            except:
                pass

        jobs_data.append({
            "Display Name": j.display_name[:30],
            "Status": j.status,
            "Created": created.strftime("%Y-%m-%d %H:%M"),
            "Duration": duration_str,
        })

    df = pd.DataFrame(jobs_data)
    print(df.to_string(index=False))

    # Active jobs with links
    active_jobs = [j for j in jobs if j.status in ['Running', 'Queued', 'Starting', 'Preparing', 'Finalizing']]
    if active_jobs:
        print(f"\n🔗 Active Job Links:")
        for j in active_jobs:
            print(f"  • {j.display_name}: {j.studio_url}")
else:
    print("  No training jobs found")

# === Next Steps ===
print("\n" + "="*80)
if env_build_active:
    print("⏳ NEXT: Wait for environment build (~5-15 min), then training will start")
elif job_name and status.get('status') == 'Running':
    print("✅ NEXT: Training in progress - monitor in Azure ML Studio")
elif job_name and status.get('status') == 'Preparing':
    print("⏳ NEXT: Job preparing - should start soon")
elif job_name and status.get('status') == 'Completed':
    print("✅ NEXT: Training complete - proceed to notebook 03 to download model")
else:
    print("💡 NEXT: Submit a training job in Section 5")

### Understanding Job Dependencies

When you submit a training job, Azure ML performs these steps in sequence:

1. **Environment Image Building** (`prepare_image` experiment)
   - Triggered automatically when creating/updating an environment (Section 3)
   - Builds a Docker image with PyTorch, CUDA, transformers from your `conda.yaml`
   - Runs as a separate background job
   - Typically takes 5-15 minutes

2. **Training Job Preparation** (Your experiment: `phi-4-training`)
   - Job status: "Preparing"
   - Waits for environment image build to complete
   - Allocates compute resources
   - Sets up job environment

3. **Training Execution**
   - Job status: "Running"
   - Downloads base model from Azure AI Foundry
   - Executes training script
   - Saves checkpoints to datastore

**If your training job shows "Preparing" for >10 minutes**, check the `prepare_image` experiment - the environment build may still be in progress.

In [ ]:
# Wait for job completion (optional - can take 1-2 hours)
# Uncomment to wait for completion

# print("Waiting for job to complete...")
# print("This may take 1-2 hours depending on data size and configuration.")
# print("You can safely interrupt this cell and check status later.\n")

# final_status = job_manager.wait_for_completion(
#     job_name=job_name,
#     timeout_seconds=7200,  # 2 hours
#     check_interval=60,  # Check every minute
# )

# print(f"\n✓ Job completed with status: {final_status}")

## 7. Check Recent Training Jobs

In [ ]:
import pandas as pd

# List recent jobs
jobs = job_manager.list_jobs(experiment_name=experiment_name, max_results=5)

print(f"Recent jobs in '{experiment_name}':")
print("\n")

jobs_data = []
for j in jobs:
    jobs_data.append({
        "Name": j.name,
        "Display Name": j.display_name,
        "Status": j.status,
        "Created": j.creation_context.created_at.strftime("%Y-%m-%d %H:%M"),
    })

if jobs_data:
    df = pd.DataFrame(jobs_data)
    print(df.to_string(index=False))
else:
    print("No jobs found in this experiment.")

## Summary

✅ **Training job successfully submitted to Azure ML!**

**What's happening:**
- Job is running on Azure ML compute cluster
- Model is being downloaded from HuggingFace automatically
- Training data is mounted from blob storage
- Model checkpoints are being saved to Azure ML datastore
- Metrics are logged to Azure ML Studio

**Next steps:**
- Monitor job in Azure ML Studio (link above)
- Download trained model when complete (notebook 03-download-trained-model.ipynb)
- Or wait for job completion notification

## Troubleshooting

**Job Submission Failed**:
- Verify compute cluster exists and is running
- Check training data is uploaded to datastore
- Ensure sufficient GPU quota in subscription
- Review error details in Azure ML Studio

**"Stream not found" or "NotFound" Error**:
- Training data not uploaded to blob storage
- Run notebook `01-prepare-data.ipynb` to upload data
- Use the data verification cell (Section 5) to confirm files exist
- Check that files are in `training-data/train.jsonl` and `training-data/val.jsonl` paths

**Job Queued for Long Time**:
- Compute cluster may be scaling up from 0 nodes
- Check cluster status in Azure ML Studio
- Verify max_instances > 0 in cluster config

**Training Errors**:
- Check job logs in Azure ML Studio → Jobs → [job name] → Outputs + logs
- Common issues: CUDA out of memory, data path errors, dependency conflicts, model download failures
- Reduce batch size if OOM errors occur

**Slow Training**:
- Verify GPU is being utilized (check metrics in Studio)
- Consider using larger VM size (more GPUs)
- Enable mixed precision training if not already enabled

---

## Navigation

**Previous:** [01-prepare-data.ipynb](01-prepare-data.ipynb) | **Next:** [03-download-trained-model.ipynb](03-download-trained-model.ipynb)